In [ ]:
from konlpy.tag import Okt

okt = Okt()

text = "나는 오늘 카페에서 공부했습니다."

print("형태소 : ", okt.morphs(text))
print("명사 : ", okt.nouns(text))
print("품사 : ", okt.pos(text))

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

text = "나는 오늘 카페에서 공부했습니다."

result = tokenizer(text)

print(result)

In [ ]:
import numpy as np

sentence = "나는 밥을 먹고 싶다"

words = sentence.split()
print("words : ", words)

vocab = list(dict.fromkeys(words))
print()
print("vocab : ", vocab)

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
print()
print("word2idx : ", word2idx)
print()
print("idx2word : ", idx2word)

sentence_indices = [word2idx[word] for word in words]
print()
print("sentence_indices : ", sentence_indices)

restored_sentence = [idx2word[index] for index in sentence_indices]
print()
print("restored_sentence : ", restored_sentence)


def one_hot(index, vocab_size):
    vector = np.zeros(vocab_size)

    vector[index] = 1

    return vector


vocab_size = len(vocab)
print()
print("vocab_size : ", vocab_size)

one_hot_vectors = []

for index in sentence_indices:
    vector = one_hot(index, vocab_size)
    one_hot_vectors.append(vector)

print()
print("one_hot_vectors : ", one_hot_vectors)

one_hot_vectors = np.array(one_hot_vectors)

X = sentence_indices[:-1]
Y = sentence_indices[1:]

print()
print("X : ", X)
print()
print("y : ", Y)

print()
for x, y in zip(X, Y):
    print(f"{idx2word[x]} -> {idx2word[y]}")

X_one_hot = np.array([one_hot(index, vocab_size) for index in X])
print()
print("X_one_hot : ", X_one_hot)


for i in range(len(X)):
    print(f"\nTime Step {i + 1}")

    print(f"입력 단어 : {idx2word[X[i]]}")
    print(f"입력 숫자 : {X[i]}")
    print(f"입력 벡터 : {X_one_hot[i]}")

    print(f"정답 단어 : {idx2word[Y[i]]}")
    print(f"정답 숫자 : {Y[i]}")

In [ ]:
import numpy as np

sentence = "나는 밥을 먹고 싶다"
words = sentence.split()
vocab = list(dict.fromkeys(words))

word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}


def one_hot(index, vocab_size):
    vector = np.zeros(vocab_size)

    vector[index] = 1
    return vector


indices = [word2idx[word] for word in words]


X = indices[:-1]
Y = indices[1:]


class RNN:
    def __init__(self, input_size, hidden_size, vocab_size):
        self.W_x = np.random.randn(input_size, hidden_size) * 0.01
        self.W_h = np.random.randn(hidden_size, hidden_size) * 0.01
        self.W_y = np.random.randn(hidden_size, vocab_size) * 0.01
        self.b_h = np.zeros(hidden_size)
        self.b_y = np.zeros(vocab_size)

    def forward(self, x, h_prev):
        h = np.tanh(x @ self.W_x + h_prev @ self.W_h + self.b_h)
        logits = h @ self.W_y + self.b_y

        return h, logits


vocab_size = len(vocab)

hidden_size = 18

rnn = RNN(input_size=vocab_size, hidden_size=hidden_size, vocab_size=vocab_size)

h = np.zeros(hidden_size)

for t in range(len(X)):
    x_word = idx2word[X[t]]

    x = one_hot(X[t], vocab_size)

    h, logits = rnn.forward(x, h)

    predicted_index = np.argmax(logits)

    predicted_word = idx2word[predicted_index]

    target_word = idx2word[Y[t]]

    print("=" * 50)

    print(f"Time Step : {t + 1}")

    print(f"입력       : {x_word}")

    print(f"정답       : {target_word}")

    print(f"현재 예측   : {predicted_word}")

    print()

    print("현재 Hidden State")

    print(h)

In [ ]:
from datasets import load_dataset
from kiwipiepy import Kiwi
from collections import Counter

dataset = load_dataset("FISA-conclave/news-sentiment-dataset")

kiwi = Kiwi()
counter = Counter()

for text in dataset["train"]["sentence"]:
    tokens = kiwi.tokenize(text)

    for token in tokens:
        counter[token.form] += 1

print(counter.most_common(20))

VOCAB_SIZE = 20000

word2idx = {"<PAD>": 0, "<UNK>": 1}

for word, count in counter.most_common(VOCAB_SIZE - 2):
    word2idx[word] = len(word2idx)


def text_to_indices(text):
    tokens = kiwi.tokenize(text)

    indices = []

    for token in tokens:
        word = token.form

        if word in word2idx:
            indices.append(word2idx[word])
        else:
            indices.append(word2idx["<UNK>"])
    return indices

In [ ]:
import torch
from torch.utils.data import Dataset

label2idx = {"negative": 0, "neutral": 1, "positive": 2}

idx2label = {0: "negative", 1: "neutral", 2: "positive"}


class NewsDataset(Dataset):
    def __init__(self, dataset_split):
        self.texts = dataset_split["sentence"]
        self.labels = dataset_split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = text_to_indices(self.texts[idx])
        y = label2idx[self.labels[idx]]

        return x, y


train_dataset = NewsDataset(dataset["train"])
test_dataset = NewsDataset(dataset["test"])

X, y = train_dataset[0]
print("X : ", X)
print()
print("y : ", y)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

PAD_IDX = word2idx["<PAD>"]


def collate_fn(batch):
    xs, ys = zip(*batch)

    xs = [torch.tensor(x, dtype=torch.long) for x in xs]

    xs = pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)

    ys = torch.tensor(ys, dtype=torch.long)

    return xs, ys

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from datasets import load_dataset
from kiwipiepy import Kiwi
from collections import Counter


dataset = load_dataset("FISA-conclave/news-sentiment-dataset")

kiwi = Kiwi()

counter = Counter()

for text in dataset["train"]["sentence"]:
    tokens = kiwi.tokenize(text)

    for token in tokens:
        counter[token.form] += 1

In [ ]:
vocab_size = len(counter)

word2idx = {"<PAD>": 0, "<UNK>": 1}

for word, count in counter.most_common(vocab_size - 2):
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}


def text_to_indices(text):
    tokens = kiwi.tokenize(text)

    indices = []

    for token in tokens:
        word = token.form

        if word in word2idx:
            indices.append(word2idx[word])
        else:
            indices.append(word2idx["<UNK>"])
    return indices


label2idx = {"negative": 0, "neutral": 1, "positive": 2}

idx2label = {0: "negative", 1: "neutral", 2: "positive"}


class NewsDataset(Dataset):
    def __init__(self, dataset_split):
        self.texts = dataset_split["sentence"]
        self.labels = dataset_split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = text_to_indices(self.texts[idx])

        y = label2idx[self.labels[idx]]

        return x, y


train_dataset = NewsDataset(dataset["train"])

test_dataset = NewsDataset(dataset["test"])

PAD_IDX = word2idx["<PAD>"]


def collate_fn(batch):
    xs, ys = zip(*batch)

    xs = [torch.tensor(x, dtype=torch.long) for x in xs]

    xs = pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)

    ys = torch.tensor(ys, dtype=torch.long)

    return xs, ys


train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn
)

In [ ]:
import torch
import torch.nn as nn


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=PAD_IDX
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.rnn(x)

        last_hidden = hidden[-1]

        logits = self.fc(last_hidden)

        return logits


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("사용 장치:", device)

VOCAB_SIZE = len(word2idx)

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_CLASSES = 3

model = RNNClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
)

model = model.to(device)

In [ ]:
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from datasets import load_dataset
from kiwipiepy import Kiwi
from collections import Counter


dataset = load_dataset("FISA-conclave/news-sentiment-dataset")

kiwi = Kiwi()

counter = Counter()

for text in dataset["train"]["sentence"]:
    tokens = kiwi.tokenize(text)

    for token in tokens:
        counter[token.form] += 1

vocab_size = len(counter)

word2idx = {"<PAD>": 0, "<UNK>": 1}

for word, count in counter.most_common(vocab_size - 2):
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}


def text_to_indices(text):
    tokens = kiwi.tokenize(text)

    indices = []

    for token in tokens:
        word = token.form

        if word in word2idx:
            indices.append(word2idx[word])
        else:
            indices.append(word2idx["<UNK>"])
    return indices


label2idx = {"negative": 0, "neutral": 1, "positive": 2}

idx2label = {0: "negative", 1: "neutral", 2: "positive"}


class NewsDataset(Dataset):
    def __init__(self, dataset_split):
        self.texts = dataset_split["sentence"]
        self.labels = dataset_split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = text_to_indices(self.texts[idx])

        y = label2idx[self.labels[idx]]

        return x, y


train_dataset = NewsDataset(dataset["train"])

test_dataset = NewsDataset(dataset["test"])

PAD_IDX = word2idx["<PAD>"]


def collate_fn(batch):
    xs, ys = zip(*batch)

    xs = [torch.tensor(x, dtype=torch.long) for x in xs]

    xs = pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)

    ys = torch.tensor(ys, dtype=torch.long)

    return xs, ys


train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn
)


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=PAD_IDX
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.rnn(x)

        last_hidden = hidden[-1]

        logits = self.fc(last_hidden)

        return logits


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("사용 장치:", device)

VOCAB_SIZE = len(word2idx)

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_CLASSES = 3

model = RNNClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
)

model = model.to(device)

EPOCHS = 5

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(x_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)

        correct += (predictions == y_batch).sum().item()

        total += y_batch.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch [{epoch + 1}/{EPOCHS}] Loss: {avg_loss:.4f} Accuracy: {accuracy:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# 1. 아주 작은 번역 데이터
# ============================================================

pairs = [
    ("나는 먹는다", "i eat"),
    ("나는 잔다", "i sleep"),
    ("나는 달린다", "i run"),
    ("너는 먹는다", "you eat"),
    ("너는 잔다", "you sleep"),
    ("너는 달린다", "you run"),
]


# ============================================================
# 2. 한국어 / 영어 Vocabulary 만들기
# ============================================================

korean_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "나는": 3,
    "너는": 4,
    "먹는다": 5,
    "잔다": 6,
    "달린다": 7,
}

english_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "i": 3,
    "you": 4,
    "eat": 5,
    "sleep": 6,
    "run": 7,
}

english_idx2word = {value: key for key, value in english_vocab.items()}


# ============================================================
# 3. 문장을 Token ID로 변환
# ============================================================


def korean_to_ids(sentence):
    tokens = sentence.split()

    return [korean_vocab[token] for token in tokens]


def english_to_ids(sentence):
    tokens = sentence.split()

    return [
        english_vocab["<SOS>"],
        *[english_vocab[token] for token in tokens],
        english_vocab["<EOS>"],
    ]


# ============================================================
# 4. Encoder
# ============================================================


class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):

        super().__init__()

        # 한국어 Token ID → 한국어 벡터
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 한국어 벡터 → hidden state
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, x):

        # x:
        # [한국어 Token ID]
        #
        # 예:
        # [3, 5]
        #
        # 나는 = 3
        # 먹는다 = 5

        embedded = self.embedding(x)

        # embedded:
        # [2개의 단어, embedding_dim]

        outputs, hidden = self.rnn(embedded)

        # hidden:
        # Encoder가 한국어 문장을 읽고
        # 마지막에 만든 기억

        return hidden


# ============================================================
# 5. Decoder
# ============================================================


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):

        super().__init__()

        # 영어 Token ID → 영어 벡터
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 영어 벡터 + Encoder의 hidden
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        # hidden → 영어 단어 점수
        #
        # 영어 단어가 8개이므로
        # 출력도 8개
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):

        # x:
        # 현재 Decoder에 들어가는 영어 단어
        #
        # 예:
        # <SOS>

        embedded = self.embedding(x)

        output, hidden = self.rnn(embedded, hidden)

        # hidden state를
        # 영어 단어 8개의 점수로 변환

        prediction = self.fc(output)

        return prediction, hidden


# ============================================================
# 6. Encoder + Decoder
# ============================================================

embedding_dim = 16
hidden_dim = 32

encoder = Encoder(
    vocab_size=len(korean_vocab), embedding_dim=embedding_dim, hidden_dim=hidden_dim
)

decoder = Decoder(
    vocab_size=len(english_vocab), embedding_dim=embedding_dim, hidden_dim=hidden_dim
)


# ============================================================
# 7. Loss / Optimizer
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)


# ============================================================
# 8. 학습
# ============================================================

for epoch in range(1000):
    total_loss = 0

    for korean, english in pairs:
        # ----------------------------------------------------
        # 한국어
        # ----------------------------------------------------

        korean_ids = korean_to_ids(korean)

        korean_tensor = torch.tensor(korean_ids, dtype=torch.long).unsqueeze(0)

        # 예:
        #
        # "나는 먹는다"
        #
        # ↓
        #
        # [3, 5]

        # ----------------------------------------------------
        # 영어
        # ----------------------------------------------------

        english_ids = english_to_ids(english)

        english_tensor = torch.tensor(english_ids, dtype=torch.long).unsqueeze(0)

        # 예:
        #
        # "i eat"
        #
        # ↓
        #
        # [<SOS>, i, eat, <EOS>]

        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        hidden = encoder(korean_tensor)

        # 한국어
        #
        # 나는 → 먹는다
        #
        # ↓
        #
        # Encoder
        #
        # ↓
        #
        # hidden

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        decoder_input = english_tensor[:, 0:1]
        # 처음에는 <SOS>

        loss = 0

        for t in range(1, english_tensor.size(1)):
            output, hidden = decoder(decoder_input, hidden)

            # output:
            #
            # 영어 단어 8개의 점수
            #
            # [<PAD>, <SOS>, <EOS>, i, you, eat, sleep, run]

            target = english_tensor[:, t]

            # 정답 영어 단어

            loss += criterion(output.squeeze(1), target)

            # 학습할 때는
            # 정답 단어를 다음 입력으로 사용

            decoder_input = target.unsqueeze(1)

        # ----------------------------------------------------
        # 역전파
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")


# ============================================================
# 9. 번역 함수
# ============================================================


def translate(sentence):

    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        # ----------------------------------------------------
        # 한국어 → Token ID
        # ----------------------------------------------------

        korean_ids = korean_to_ids(sentence)

        korean_tensor = torch.tensor(korean_ids, dtype=torch.long).unsqueeze(0)

        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        hidden = encoder(korean_tensor)

        # ----------------------------------------------------
        # Decoder 시작
        # ----------------------------------------------------

        decoder_input = torch.tensor([[english_vocab["<SOS>"]]], dtype=torch.long)

        result = []

        # 최대 10개의 영어 단어 생성

        for _ in range(10):
            output, hidden = decoder(decoder_input, hidden)

            # 가장 높은 점수를 가진 단어 선택

            next_token = output.argmax(dim=-1).item()

            # EOS면 종료

            if next_token == english_vocab["<EOS>"]:
                break

            # 영어 단어로 변환

            word = english_idx2word[next_token]

            result.append(word)

            # 방금 만든 단어를
            # 다음 입력으로 사용

            decoder_input = torch.tensor([[next_token]], dtype=torch.long)

        return " ".join(result)


# ============================================================
# 10. 실제 번역 테스트
# ============================================================

print()
print("번역 결과")
print("--------------------")

test_sentences = [
    "나는 먹는다",
    "나는 잔다",
    "나는 달린다",
    "너는 먹는다",
    "너는 잔다",
    "너는 달린다",
]

for sentence in test_sentences:
    result = translate(sentence)

    print(f"{sentence} → {result}")

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


np.random.seed(42)

n_normal = 1000

normal_data = np.column_stack(
    [
        np.random.normal(50000, 15000, n_normal),  # 월 구매금액
        np.random.normal(6, 2, n_normal),  # 월 구매횟수
        np.random.normal(20, 5, n_normal),  # 평균 구매간격
        np.random.normal(1, 0.5, n_normal),  # 반품횟수
    ]
)

normal_df = pd.DataFrame(
    normal_data,
    columns=["monthly_amount", "purchase_count", "avg_interval", "return_count"],
)

n_anomaly = 100

anomaly_data = np.column_stack(
    [
        np.random.normal(5000000, 1000000, n_anomaly),
        np.random.normal(200, 50, n_anomaly),
        np.random.normal(0.5, 0.2, n_anomaly),
        np.random.normal(100, 20, n_anomaly),
    ]
)

anomaly_df = pd.DataFrame(
    anomaly_data,
    columns=["monthly_amount", "purchase_count", "avg_interval", "return_count"],
)

scaler = StandardScaler()

X_train = scaler.fit_transform(normal_df)
X_anomaly = scaler.transform(anomaly_df)


class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))

        self.decoder = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 4))

    def forward(self, x):
        z = self.encoder(x)

        x_hat = self.decoder(z)

        return x_hat


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
dataset = TensorDataset(X_train_tensor)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = AutoEncoder()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in range(epochs):
    total_loss = 0

    for (x,) in loader:
        x_hat = model(x)

        loss = criterion(x_hat, x)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d} Loss: {total_loss / len(loader):.4f}")


model.eval()
with torch.no_grad():
    normal_tensor = torch.tensor(X_train[:10], dtype=torch.float32)

    normal_reconstructed = model(normal_tensor)

    normal_errors = torch.mean((normal_tensor - normal_reconstructed) ** 2, dim=1)

    print(normal_errors)

anomaly_tensor = torch.tensor(X_anomaly, dtype=torch.float32)

with torch.no_grad():
    anomaly_reconstructed = model(anomaly_tensor)

    anomaly_errors = torch.mean((anomaly_tensor - anomaly_reconstructed) ** 2, dim=1)

    print(anomaly_errors[:10])

Epoch  10 Loss: 0.6063
Epoch  20 Loss: 0.5132
Epoch  30 Loss: 0.5059
Epoch  40 Loss: 0.4835
Epoch  50 Loss: 0.4758
Epoch  60 Loss: 0.4548
Epoch  70 Loss: 0.4401
Epoch  80 Loss: 0.4355
Epoch  90 Loss: 0.4245
Epoch 100 Loss: 0.4303
tensor([0.7346, 0.0572, 0.0474, 0.0695, 0.1424, 0.0926, 0.3638, 0.4068, 0.7038,
        0.0747])
tensor([ 7249.3125, 13501.4492, 16049.7334, 15389.7285,  7619.7080, 19242.9707,
         3348.4800, 10716.5430, 10006.8789, 17349.0820])


In [ ]:
import random
import re

import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

dataset = load_dataset("Helsinki-NLP/opus-100", "en-ko")

for i in range(5):
    print(dataset["train"][i])

TRAIN_SIZE = 20000
VALID_SIZE = 2000
TEST_SIZE = 500

train_data = dataset["train"].select(range(min(TRAIN_SIZE, len(dataset["train"]))))

val_data = dataset["validation"].select(
    range(min(TRAIN_SIZE, len(dataset["validation"])))
)

test_data = dataset["test"].select(range(min(TRAIN_SIZE, len(dataset["test"]))))


def normalize_text(text):
    text = text.strip()

    text = re.sub(r"\s+", " ", text)

    return text


PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"

SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]


def build_vocab(data, language):
    counter = {}
    for item in data:
        sentence = normalize_text(item["translation"][language])

        for char in sentence:
            counter[char] = counter.get(char, 0) + 1

    vocab = SPECIAL_TOKENS.copy()

    for char in counter:
        if char not in vocab:
            vocab.append(char)

    stoi = {token: idx for idx, token in enumerate(vocab)}

    itos = {idx: token for idx, token in enumerate(vocab)}

    return stoi, itos


ko_stoi, ko_itos = build_vocab(train_data, "ko")
en_stoi, en_itos = build_vocab(train_data, "en")


def tokenize_sentence(sentence, stoi, max_length=50):
    sentence = normalize_text(sentence)

    tokens = [SOS_TOKEN]
    tokens += list(sentence)
    tokens += [EOS_TOKEN]
    tokens += tokens[:max_length]
    ids = []

    for token in tokens:
        if token in stoi:
            ids.append(stoi[token])
        else:
            ids.append(stoi[token])

Generating validation split: 100%|██████████| 2000/2000 [00:00<00:00, 1098416.66 examples/s]


{'translation': {'en': "They're shaped like a bus.", 'ko': '할머니처럼 만들었지만.. ? 엉망이지만..'}}
{'translation': {'en': "I ain't fishing' 'em out.", 'ko': '그거 꺼내려다가는'}}
{'translation': {'en': "You are torturing god's creatures in an age where we have the technology that no longer requires us to.", 'ko': '선생님은 이 기술력이 있는 시대에 그러지 않아도 되는데도 신의 피조물을 괴롭히고 있다고요'}}
{'translation': {'en': 'Roger that.', 'ko': '아무도 없음. 알았다 오바.'}}
{'translation': {'en': 'How could my father let him do this?', 'ko': '어떻게 아빠가 저걸 허락할 수가 있지?'}}
한국어 Vocabulary 크기: 1534
영어 Vocabulary 크기: 127
